In [23]:
"""
IFRU Baseline for WBPR-LightGCN
================================================================================
论文：Recommendation Unlearning via Influence Function
     (Yang Zhang et al., ACM Trans. Recomm. Syst. 2024)

论文方程对应（严格遵循）：
  Eq.7-9     : L_d + L_s 形式化（直接影响 + 溢出影响）
  Eq.12-13   : I = I_d + I_s = -H^{-1}(∇L_d + ∇L_s)；t* = -(∇L_d + ∇L_s)
  Eq.14      : 遗忘模型 θ' = θ̂ - (1/|D|) * I(θ̂; D_r)
  Eq.15      : H^{-1}t* 转化为 min_t (1/2) t^T H t - t^T t*，用 Adam 求解
  Eq.16      : HVP：H_θ t = ∇_θ(∇_θ L^T t)
  Algorithm 1: 重要性剪枝（0阶/1阶邻居得分）
  Section 3.3: LightGCN 实例化

关键实现说明：
  (1) L_d / L_s 使用 .sum()（对应 Eq.7/9 的 Σ 定义）
      HVP mini-batch loss 使用 .mean()（对应 Eq.1 的 (1/|D|)Σ）

  (2) HVP 实现方式：
      torch.sparse.mm 不支持二阶 autograd（create_graph 后第二次 .grad 返回 0）。
      利用 LightGCN 传播的线性性：E_final = M E_0，则
        H_{φ} v = M^T (H_{E_final} (M v))
      其中 M 对称（hat_A 对称），M^T = M。
      三步分解：
        Step 1: δE = M·scatter(v) — 无 autograd 的 sparse mm
        Step 2: H_{E_final}·δE — 在稠密 E_final 空间用 create_graph，无 sparse mm
        Step 3: M·result — 无 autograd 的 sparse mm
      create_graph 只施加于稠密 BPR loss，完全绕开 sparse mm 的二阶限制。

  (3) D_c 覆盖 1 跳（Section 4.3："for simplicity, K=1"），
      与 N_LAYERS=3 无关，论文自身的简化。
================================================================================
"""

import os, time, random
import numpy as np
import pandas as pd
import torch
import torch.autograd as autograd
import scipy.sparse as sp
from collections import defaultdict

# ══════════════════════════════════════════════════════════════════════════════
# 0. 配置（路径 & 超参）
# ══════════════════════════════════════════════════════════════════════════════
BASE = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR"

DATA_PATH      = "/Users/yubinhao/ml-1m/Amazon Review/数据集/Amazon_Reviews_Processed"
MODEL_PATH     = f"{BASE}/模型文件/model_best.pt"
EMBED_PATH     = f"{BASE}/模型文件/embeddings_best.pt"
MAPPING_PATH   = f"{BASE}/模型文件/id_mappings.pt"
ITEM_SETS_PATH = f"{BASE}/pattern_basis/pattern_item_sets.pt"
BASIS_PATH     = f"{BASE}/pattern_basis/pattern_basis.pt"
INFLUENCE_PATH = f"{BASE}/pattern_influence/influence_scores.pt"
THRESHOLD_PATH = f"{BASE}/遗忘实验/threshold_info.pt"
UNLEARN_PATH   = f"{BASE}/遗忘实验/embeddings_unlearned.pt"
RETRAIN_PATH   = f"{BASE}/Retrain/embeddings_retrain.pt"
SAVE_DIR       = f"{BASE}/IFRU"
os.makedirs(SAVE_DIR, exist_ok=True)

# 模型超参（与 LightGCN加权BPR.ipynb 完全一致）
EMBED_DIM        = 64
N_LAYERS         = 3
RATING_MAX       = 5.0
MIN_INTERACTIONS = 5
SEED             = 42

# IFRU 超参（均取自论文 Section 4.1.4 默认值）
PRUNE_A0       = 1.0
PRUNE_A1       = 0.6
LR_INFLUENCE   = 1   # 论文候选：{1e3,2e3,5e3,1e4,2e4,5e4}
N_ITERS        = 100
N_HVP_BATCHES  = 3
HVP_BATCH_SIZE = 2048
DAMP           = 0.01 

TOP_K  = 8
EPS    = 1e-10
MIN_NT = 0.01

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("[配置] 完成")


# ══════════════════════════════════════════════════════════════════════════════
# 1. 数据加载
# ══════════════════════════════════════════════════════════════════════════════
print("\n[1] 加载数据...")

model_ckpt = torch.load(MODEL_PATH,   map_location="cpu", weights_only=False)
mapping    = torch.load(MAPPING_PATH, map_location="cpu", weights_only=False)
n_users    = len(mapping["user2idx"])
n_items    = len(mapping["item2idx"])
d          = EMBED_DIM

hat_user_raw = model_ckpt["model_state_dict"]["user_emb.weight"].float()
hat_item_raw = model_ckpt["model_state_dict"]["item_emb.weight"].float()

orig_emb          = torch.load(EMBED_PATH, map_location="cpu", weights_only=False)
user_emb_orig_agg = orig_emb["user_embeddings"].numpy()
item_emb_orig_agg = orig_emb["item_embeddings"].numpy()

df = pd.read_csv(DATA_PATH, usecols=["user_id", "item_id", "rating"])
uc = df.groupby("user_id").size()
df = df[df["user_id"].isin(uc[uc >= MIN_INTERACTIONS].index)].copy()
df["uid"] = df["user_id"].map(mapping["user2idx"])
df["iid"] = df["item_id"].map(mapping["item2idx"])
df.dropna(subset=["uid", "iid"], inplace=True)
df["uid"] = df["uid"].astype(int)
df["iid"] = df["iid"].astype(int)
n_train      = len(df)
all_uids_arr = df["uid"].values
all_iids_arr = df["iid"].values

user_pos = defaultdict(set)
item_pos = defaultdict(set)
for r in df.itertuples(index=False):
    user_pos[r.uid].add(r.iid)
    item_pos[r.iid].add(r.uid)

infl   = torch.load(INFLUENCE_PATH, map_location="cpu", weights_only=False)
thresh = torch.load(THRESHOLD_PATH, map_location="cpu", weights_only=False)
sets   = torch.load(ITEM_SETS_PATH, map_location="cpu", weights_only=False)
basis  = torch.load(BASIS_PATH,     map_location="cpu", weights_only=False)
udata  = torch.load(UNLEARN_PATH,   map_location="cpu", weights_only=False)
rdata  = torch.load(RETRAIN_PATH,   map_location="cpu", weights_only=False)

influence_matrix  = infl["influence_matrix"]
sampled_uids      = infl["sampled_uids"]
hq_patterns       = infl["hq_patterns"]
tau_u             = thresh["tau_u"]
pattern_item_sets = sets["pattern_item_sets"]
pattern_basis_np  = basis["pattern_basis"].numpy()

user_emb_unlearn = udata["user_emb_after"]
user_emb_retrain = rdata["user_embeddings"].numpy()
item_emb_retrain = rdata["item_embeddings"].numpy()

print(f"  n_users={n_users}, n_items={n_items}, n_train={n_train}")


# ══════════════════════════════════════════════════════════════════════════════
# 2. 邻接矩阵构建（与训练代码完全一致）
# ══════════════════════════════════════════════════════════════════════════════
def build_adj(df_local, n_u, n_i, rmax=5.0):
    u = df_local["uid"].values.astype(np.int64)
    i = df_local["iid"].values.astype(np.int64)
    w = (df_local["rating"].values / rmax).astype(np.float32)
    N = n_u + n_i
    row = np.concatenate([u,       i + n_u])
    col = np.concatenate([i + n_u, u      ])
    val = np.concatenate([w,       w      ])
    A  = sp.csr_matrix((val, (row, col)), shape=(N, N), dtype=np.float32)
    dw = np.array(A.sum(1)).flatten()
    ds = np.where(dw > 0, 1.0 / np.sqrt(dw), 0.0)
    H  = (sp.diags(ds) @ A @ sp.diags(ds)).tocoo().astype(np.float32)
    idx = torch.LongTensor(np.vstack([H.row, H.col]))
    return torch.sparse_coo_tensor(idx, torch.FloatTensor(H.data), (N, N)).coalesce()

print("\n[2] 构建 hat_A(D)...")
hat_A = build_adj(df, n_users, n_items, RATING_MAX)
print(f"  hat_A: nnz={hat_A._nnz()}")


# ══════════════════════════════════════════════════════════════════════════════
# 3. 构建 D_remove（与重训练实验.ipynb 完全一致）
# ══════════════════════════════════════════════════════════════════════════════
print("\n[3] 构建 D_remove...")

S_u_dict = {}
for idx2, uid in enumerate(sampled_uids):
    scores    = influence_matrix[idx2]
    threshold = tau_u[idx2]
    S_u_dict[uid] = [hq_patterns[ki] for ki, s in enumerate(scores) if s > threshold]

D_remove_set = set()
for uid, pats in S_u_dict.items():
    Bu = user_pos.get(uid, set())
    for k in pats:
        for iid in Bu & set(pattern_item_sets[k]):
            D_remove_set.add((uid, iid))

V_Dr_u = set(u for u, i in D_remove_set)
V_Dr_i = set(i for u, i in D_remove_set)
print(f"  |D_remove|={len(D_remove_set)}, |V(D_r)|: {len(V_Dr_u)} 用户 / {len(V_Dr_i)} 物品")


# ══════════════════════════════════════════════════════════════════════════════
# 4. 构建 hat_A_{-r} 与 D_c（论文 Section 3.3）
# ══════════════════════════════════════════════════════════════════════════════
print("\n[4] 构建 hat_A_{-r} 与 D_c...")

remove_df   = pd.DataFrame(list(D_remove_set), columns=["uid", "iid"])
remove_df["_rm"] = True
df_merged   = df.merge(remove_df, on=["uid", "iid"], how="left")
df_retrain_data = df_merged[df_merged["_rm"].isna()].drop(columns=["_rm"]).copy()
n_retrain   = len(df_retrain_data)

hat_A_minus_r = build_adj(df_retrain_data, n_users, n_items, RATING_MAX)

Dc_mask = df_retrain_data["uid"].isin(V_Dr_u) | df_retrain_data["iid"].isin(V_Dr_i)
df_Dc   = df_retrain_data[Dc_mask].copy()
Dc_uids = df_Dc["uid"].values.astype(np.int64)
Dc_iids = df_Dc["iid"].values.astype(np.int64)
print(f"  |D_retrain|={n_retrain}, |D_c|={len(df_Dc)}")


# ══════════════════════════════════════════════════════════════════════════════
# 工具函数
# ══════════════════════════════════════════════════════════════════════════════
def sample_neg_exact(uids_arr, user_pos_dict_, n_i, rng):
    neg = rng.integers(0, n_i, len(uids_arr))
    for j, u in enumerate(uids_arr):
        while int(neg[j]) in user_pos_dict_[int(u)]:
            neg[j] = rng.integers(0, n_i)
    return neg.astype(np.int64)


def bpr_loss_sum(u_emb, pos_emb, neg_emb):
    """求和形式，用于 L_d（Eq.7）和 L_s（Eq.9）。"""
    return -torch.log(
        torch.sigmoid((u_emb * pos_emb).sum(-1) - (u_emb * neg_emb).sum(-1)) + 1e-8
    ).sum()


def bpr_loss_mean(u_emb, pos_emb, neg_emb):
    """均值形式，用于 HVP Hessian 估计，对应 Eq.1 的 (1/|D|) Σ。"""
    return -torch.log(
        torch.sigmoid((u_emb * pos_emb).sum(-1) - (u_emb * neg_emb).sum(-1)) + 1e-8
    ).mean()


def make_E0_with_phi(phi_u, phi_i, phi_u_idx_t, phi_i_idx_t, frozen_u, frozen_i):
    """构建含梯度的 E_0，用于计算 t*（Eq.13）。"""
    phi_full_u = torch.zeros(n_users, d).index_put((phi_u_idx_t,), phi_u)
    phi_full_i = torch.zeros(n_items, d).index_put((phi_i_idx_t,), phi_i)
    return frozen_u + phi_full_u, frozen_i + phi_full_i


def lgcn_forward(phi_u, phi_i, phi_u_idx_t, phi_i_idx_t,
                  frozen_u, frozen_i, hat_A_, n_u, n_layers):
    """含 autograd 的 LightGCN 前向（用于一阶梯度 t*），CPU 稀疏乘法。"""
    uE0, iE0 = make_E0_with_phi(phi_u, phi_i, phi_u_idx_t, phi_i_idx_t,
                                 frozen_u, frozen_i)
    E = torch.cat([uE0, iE0], dim=0)
    layers = [E]
    for _ in range(n_layers):
        E = torch.sparse.mm(hat_A_, E)
        layers.append(E)
    E_final = torch.stack(layers, dim=1).mean(dim=1)
    return E_final[:n_u], E_final[n_u:]


def lgcn_propagate_nograd(E_in, hat_A_, n_layers_):
    """
    无 autograd 的 LightGCN 传播（用于 HVP 分解）。
    对应线性映射 M = (1/(L+1)) Σ_{l=0}^L hat_A^l。
    hat_A 对称，故 M^T = M。
    """
    with torch.no_grad():
        E = E_in
        layers = [E]
        for _ in range(n_layers_):
            E = torch.sparse.mm(hat_A_, E)
            layers.append(E)
        return torch.stack(layers, dim=1).mean(dim=1)


# ══════════════════════════════════════════════════════════════════════════════
# 5. Algorithm 1：重要性剪枝（论文 Section 3.2）
# ══════════════════════════════════════════════════════════════════════════════
print("\n[5] 重要性剪枝（论文 Algorithm 1）...")

s0_u, s0_i = defaultdict(float), defaultdict(float)
S0_u_full, S0_i_full = set(), set()
for (uid, iid) in D_remove_set:
    Nu = max(len(user_pos.get(uid, set())), 1)
    Ni = max(len(item_pos.get(iid, set())), 1)
    s0_u[uid] += 1.0 / Nu
    s0_i[iid] += 1.0 / Ni
    S0_u_full.add(uid); S0_i_full.add(iid)

def topk_by_score(score_dict, full_set, a):
    ranked = sorted(full_set, key=lambda v: score_dict.get(v, 0.0), reverse=True)
    return set(ranked[:max(1, int(len(ranked) * a))])

S0_u_keep = topk_by_score(s0_u, S0_u_full, PRUNE_A0)
S0_i_keep = topk_by_score(s0_i, S0_i_full, PRUNE_A0)

s1_u, s1_i = defaultdict(float), defaultdict(float)
S1_u_full, S1_i_full = set(), set()
for iid in S0_i_keep:
    for uid2 in item_pos.get(iid, set()):
        if uid2 not in S0_u_keep:
            s1_u[uid2] += s0_i[iid] / max(len(user_pos.get(uid2, set())), 1)
            S1_u_full.add(uid2)
for uid in S0_u_keep:
    for iid2 in user_pos.get(uid, set()):
        if iid2 not in S0_i_keep:
            s1_i[iid2] += s0_u[uid] / max(len(item_pos.get(iid2, set())), 1)
            S1_i_full.add(iid2)

S1_u_keep = topk_by_score(s1_u, S1_u_full, PRUNE_A1)
S1_i_keep = topk_by_score(s1_i, S1_i_full, PRUNE_A1)

phi_user_idx = sorted(S0_u_keep | S1_u_keep)
phi_item_idx = sorted(S0_i_keep | S1_i_keep)
n_phi_u, n_phi_i = len(phi_user_idx), len(phi_item_idx)
len_phi = (n_phi_u + n_phi_i) * d
phi_u_idx_t = torch.tensor(phi_user_idx, dtype=torch.long)
phi_i_idx_t = torch.tensor(phi_item_idx, dtype=torch.long)

print(f"  φ: {n_phi_u} 用户 + {n_phi_i} 物品 = {len_phi} 参数")

hat_phi_u = hat_user_raw[phi_user_idx].clone().detach()
hat_phi_i = hat_item_raw[phi_item_idx].clone().detach()

frozen_user = hat_user_raw.clone().detach(); frozen_user[phi_user_idx] = 0.0
frozen_item = hat_item_raw.clone().detach(); frozen_item[phi_item_idx] = 0.0


# ══════════════════════════════════════════════════════════════════════════════
# 6. 计算 t* = -∇_φ L_d - ∇_φ L_s（论文 Eq.13）
#    L_d（Eq.7）= Σ_{D_r} ℓ(·)       ← bpr_loss_sum
#    L_s（Eq.9）= Σ_{D_c} [ℓ(·;A) − ℓ(·;A_{-r})]  ← bpr_loss_sum
# ══════════════════════════════════════════════════════════════════════════════
print("\n[6] 计算 t* = -∇_φ L_d - ∇_φ L_s (Eq.13)...")
rng_tstar = np.random.default_rng(SEED)

Dr_uids = np.array([u for u, i in D_remove_set], dtype=np.int64)
Dr_iids = np.array([i for u, i in D_remove_set], dtype=np.int64)
Dr_negs = sample_neg_exact(Dr_uids, user_pos, n_items, rng_tstar)

phi_u_ld = hat_phi_u.clone().requires_grad_(True)
phi_i_ld = hat_phi_i.clone().requires_grad_(True)
uf_ld, if_ld = lgcn_forward(phi_u_ld, phi_i_ld, phi_u_idx_t, phi_i_idx_t,
                              frozen_user, frozen_item, hat_A, n_users, N_LAYERS)
L_d = bpr_loss_sum(
    uf_ld[torch.from_numpy(Dr_uids)],
    if_ld[torch.from_numpy(Dr_iids)],
    if_ld[torch.from_numpy(Dr_negs)]
)
grad_ld_u, grad_ld_i = autograd.grad(L_d, [phi_u_ld, phi_i_ld])

Dc_negs = sample_neg_exact(Dc_uids, user_pos, n_items, rng_tstar)

phi_u_ls = hat_phi_u.clone().requires_grad_(True)
phi_i_ls = hat_phi_i.clone().requires_grad_(True)
uf_D, if_D = lgcn_forward(phi_u_ls, phi_i_ls, phi_u_idx_t, phi_i_idx_t,
                            frozen_user, frozen_item, hat_A,         n_users, N_LAYERS)
uf_R, if_R = lgcn_forward(phi_u_ls, phi_i_ls, phi_u_idx_t, phi_i_idx_t,
                            frozen_user, frozen_item, hat_A_minus_r, n_users, N_LAYERS)
L_s = bpr_loss_sum(
    uf_D[torch.from_numpy(Dc_uids)],
    if_D[torch.from_numpy(Dc_iids)],
    if_D[torch.from_numpy(Dc_negs)]
) - bpr_loss_sum(
    uf_R[torch.from_numpy(Dc_uids)],
    if_R[torch.from_numpy(Dc_iids)],
    if_R[torch.from_numpy(Dc_negs)]
)
grad_ls_u, grad_ls_i = autograd.grad(L_s, [phi_u_ls, phi_i_ls])

t_star = -torch.cat([
    (grad_ld_u + grad_ls_u).detach().flatten(),
    (grad_ld_i + grad_ls_i).detach().flatten()
])
print(f"  L_d={L_d.item():.4f}, L_s={L_s.item():.4f}, ||t*||={t_star.norm().item():.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# 7. HVP + Adam 求解 I = H_φ^{-1} t*（论文 Eq.15-16）
#
#  torch.sparse.mm 不支持二阶 autograd，因此使用线性分解：
#    H_{φ} v = M^T (H_{E_final} (M v))     [M = LightGCN 传播矩阵，M^T = M]
#
#  实现步骤（每次迭代）：
#    1. scatter(t) → E_0 空间，然后无梯度地通过 LightGCN 得到 δE_final = M·t
#    2. 在稠密 E_final 空间用 create_graph 计算 H_{E_final}·δE_final
#       （仅含稠密操作，无 sparse mm，create_graph 完全有效）
#    3. 将结果再次通过 M 反向传播（M^T = M），提取 φ 分量
# ══════════════════════════════════════════════════════════════════════════════
print("\n[7] HVP + Adam 求解影响函数 (Eq.15-16)...")
print(f"  Adam lr={LR_INFLUENCE:.0e}, iters={N_ITERS}, HVP batches={N_HVP_BATCHES}×{HVP_BATCH_SIZE}")

# 预计算 E_final 在 hat_φ 处（全程固定）
with torch.no_grad():
    p_u_full = torch.zeros(n_users, d); p_u_full[phi_u_idx_t] = hat_phi_u
    p_i_full = torch.zeros(n_items, d); p_i_full[phi_i_idx_t] = hat_phi_i
    E0_hat   = torch.cat([frozen_user + p_u_full, frozen_item + p_i_full], dim=0)
    E_final_hat = lgcn_propagate_nograd(E0_hat, hat_A, N_LAYERS)  # (N, d)，固定不变

# 预采样 mini-batch（固定随机序列，可重现）
n_total_samples = N_ITERS * N_HVP_BATCHES * HVP_BATCH_SIZE
rng_hvp  = np.random.default_rng(SEED + 1)
sample_idx = rng_hvp.integers(0, n_train, n_total_samples)
pre_uids = all_uids_arr[sample_idx]
pre_pos  = all_iids_arr[sample_idx]
pre_neg  = rng_hvp.integers(0, n_items, n_total_samples).astype(np.int64)

t = torch.zeros(len_phi)
m = torch.zeros(len_phi)
v = torch.zeros(len_phi)
beta1, beta2, eps_adam = 0.9, 0.999, 1e-8

ptr = 0
t0_loop = time.time()

for it in range(1, N_ITERS + 1):

    # ── Step 1: δE_final = M · scatter(t)  [无 autograd] ─────────────────────
    t_u = t[:n_phi_u * d].reshape(n_phi_u, d)
    t_i = t[n_phi_u * d:].reshape(n_phi_i, d)

    t_full_u = torch.zeros(n_users, d); t_full_u[phi_u_idx_t] = t_u
    t_full_i = torch.zeros(n_items, d); t_full_i[phi_i_idx_t] = t_i
    t_E0     = torch.cat([t_full_u, t_full_i], dim=0)
    delta_E  = lgcn_propagate_nograd(t_E0, hat_A, N_LAYERS)  # (N, d)

    # ── Step 2: H_{E_final} · δE_final  [稠密 create_graph，无 sparse mm] ──────
    hvp_E_accum = torch.zeros_like(E_final_hat)

    for _ in range(N_HVP_BATCHES):
        b_u = pre_uids[ptr:ptr+HVP_BATCH_SIZE]
        b_p = pre_pos [ptr:ptr+HVP_BATCH_SIZE]
        b_n = pre_neg [ptr:ptr+HVP_BATCH_SIZE]
        ptr += HVP_BATCH_SIZE

        # E_final 空间的叶节点（稠密张量）
        E_leaf = E_final_hat.clone().detach().requires_grad_(True)

        # mini-batch BPR loss（仅稠密 index + dot product，无 sparse mm）
        eu  = E_leaf[torch.from_numpy(b_u)]
        ep  = E_leaf[n_users + torch.from_numpy(b_p)]
        en  = E_leaf[n_users + torch.from_numpy(b_n)]
        loss_b = bpr_loss_mean(eu, ep, en)

        # 一阶梯度，保留计算图（稠密，create_graph 完全有效）
        g_E = autograd.grad(loss_b, E_leaf, create_graph=True)[0]

        # H_{E_final} · δE_final（稠密二阶 autograd）
        hvp_E_b = autograd.grad(
            (g_E * delta_E.detach()).sum(), E_leaf
        )[0]

        hvp_E_accum += hvp_E_b.detach() / N_HVP_BATCHES

    # ── Step 3: 反向传播到 φ 空间  M^T = M（hat_A 对称）────────────────────────
    hvp_E0 = lgcn_propagate_nograd(hvp_E_accum, hat_A, N_LAYERS)  # (N, d)
    hvp_u  = hvp_E0[phi_u_idx_t]                    # (n_phi_u, d)
    hvp_i  = hvp_E0[n_users + phi_i_idx_t]          # (n_phi_i, d)
    hvp_flat = torch.cat([hvp_u.flatten(), hvp_i.flatten()])

    # ── Adam 更新（目标函数 (1/2) t^T H t - t^T t* 的梯度 = Ht - t*）─────────
    g_t = hvp_flat + DAMP * t - t_star
    m = beta1 * m + (1 - beta1) * g_t
    v = beta2 * v + (1 - beta2) * g_t * g_t
    m_hat = m / (1 - beta1 ** it)
    v_hat = v / (1 - beta2 ** it)
    t = t - LR_INFLUENCE * m_hat / (v_hat.sqrt() + eps_adam)

    if it == 1 or it % 10 == 0:
        elapsed = time.time() - t0_loop
        print(f"  Iter {it:>3d}/{N_ITERS}  ||g_t||={g_t.norm():.4f}  "
              f"||t||={t.norm():.4f}  [{elapsed:.1f}s]")

I_solved = t.detach()
print(f"\n  求解完成：||I||={I_solved.norm():.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# 8. 参数更新 θ' = θ̂ - (1/|D|) I（论文 Eq.14）& 提取 IFRU embedding
# ══════════════════════════════════════════════════════════════════════════════
print("\n[8] 参数更新 (Eq.14) 与 embedding 提取...")

I_u = I_solved[:n_phi_u * d].reshape(n_phi_u, d)
I_i = I_solved[n_phi_u * d:].reshape(n_phi_i, d)

phi_prime_u = hat_phi_u - (1.0 / n_train) * I_u
phi_prime_i = hat_phi_i - (1.0 / n_train) * I_i

theta_prime_user = hat_user_raw.clone(); theta_prime_user[phi_user_idx] = phi_prime_u
theta_prime_item = hat_item_raw.clone(); theta_prime_item[phi_item_idx] = phi_prime_i

with torch.no_grad():
    E0 = torch.cat([theta_prime_user, theta_prime_item], dim=0)
    layers = [E0]; E = E0
    for _ in range(N_LAYERS):
        E = torch.sparse.mm(hat_A_minus_r, E)
        layers.append(E)
    E_final = torch.stack(layers, dim=1).mean(dim=1)
    user_emb_ifru = E_final[:n_users].numpy()
    item_emb_ifru = E_final[n_users:].numpy()

print(f"  user_emb_ifru: {user_emb_ifru.shape}, item_emb_ifru: {item_emb_ifru.shape}")


# ══════════════════════════════════════════════════════════════════════════════
# 9. 评估（与重训练实验.ipynb 指标完全一致，新增 IFRU 列）
# ══════════════════════════════════════════════════════════════════════════════
print("\n[9] 计算评估指标...")

I_target_dict, I_nontarget_dict = {}, {}
for uid, pats in S_u_dict.items():
    Bu = user_pos.get(uid, set())
    I_union = set()
    for k in pats:
        I_union |= set(pattern_item_sets[k])
    I_target_dict[uid]    = Bu & I_union
    I_nontarget_dict[uid] = Bu - I_target_dict[uid]

valid_uids = [u for u in sampled_uids
              if len(S_u_dict[u]) > 0 and len(I_target_dict[u]) > 0]
print(f"  有效用户数: {len(valid_uids)}")

results = []
for uid in valid_uids:
    eu_orig    = user_emb_orig_agg[uid]
    eu_unlearn = user_emb_unlearn [uid]
    eu_retrain = user_emb_retrain [uid]
    eu_ifru    = user_emb_ifru    [uid]

    I_tgt = list(I_target_dict   [uid])
    I_nt  = list(I_nontarget_dict[uid])

    s_orig_t    = item_emb_orig_agg[I_tgt] @ eu_orig
    s_unlearn_t = item_emb_orig_agg[I_tgt] @ eu_unlearn
    s_retrain_t = item_emb_retrain [I_tgt] @ eu_retrain
    s_ifru_t    = item_emb_ifru    [I_tgt] @ eu_ifru
    drop_unlearn = float((s_orig_t - s_unlearn_t).mean())
    drop_retrain = float((s_orig_t - s_retrain_t).mean())
    drop_ifru    = float((s_orig_t - s_ifru_t   ).mean())

    if len(I_nt) > 0:
        s_orig_n    = item_emb_orig_agg[I_nt] @ eu_orig
        s_unlearn_n = item_emb_orig_agg[I_nt] @ eu_unlearn
        s_retrain_n = item_emb_retrain [I_nt] @ eu_retrain
        s_ifru_n    = item_emb_ifru    [I_nt] @ eu_ifru
        nt_unlearn = float(np.abs(s_orig_n - s_unlearn_n).mean())
        nt_retrain = float(np.abs(s_orig_n - s_retrain_n).mean())
        nt_ifru    = float(np.abs(s_orig_n - s_ifru_n   ).mean())
    else:
        nt_unlearn = nt_retrain = nt_ifru = 0.0

    sel_unlearn = drop_unlearn / nt_unlearn if nt_unlearn > MIN_NT else np.nan
    sel_retrain = drop_retrain / nt_retrain if nt_retrain > MIN_NT else np.nan
    sel_ifru    = drop_ifru    / nt_ifru    if nt_ifru    > MIN_NT else np.nan

    pfr_u_lst, pfr_r_lst, pfr_i_lst = [], [], []
    for k in S_u_dict[uid]:
        vk = pattern_basis_np[k]
        a_before = float(eu_orig @ vk)
        if abs(a_before) <= EPS:
            pfr_u_lst.append(0.0); pfr_r_lst.append(0.0); pfr_i_lst.append(0.0)
            continue
        pfr_u_lst.append(1.0 - float(eu_unlearn @ vk) / a_before)
        pfr_r_lst.append(1.0 - float(eu_retrain @ vk) / a_before)
        pfr_i_lst.append(1.0 - float(eu_ifru    @ vk) / a_before)
    pfr_unlearn = float(np.mean(pfr_u_lst))
    pfr_retrain = float(np.mean(pfr_r_lst))
    pfr_ifru    = float(np.mean(pfr_i_lst))

    topk_unlearn = set(np.argpartition(item_emb_orig_agg @ eu_unlearn, -TOP_K)[-TOP_K:])
    topk_retrain = set(np.argpartition(item_emb_retrain  @ eu_retrain, -TOP_K)[-TOP_K:])
    topk_ifru    = set(np.argpartition(item_emb_ifru     @ eu_ifru,    -TOP_K)[-TOP_K:])
    ov_unlearn_rt = len(topk_unlearn & topk_retrain) / TOP_K
    ov_ifru_rt    = len(topk_ifru    & topk_retrain) / TOP_K

    results.append({
        "uid":           uid,
        "drop_unlearn":  drop_unlearn, "drop_retrain": drop_retrain, "drop_ifru": drop_ifru,
        "nt_unlearn":    nt_unlearn,   "nt_retrain":   nt_retrain,   "nt_ifru":   nt_ifru,
        "sel_unlearn":   sel_unlearn,  "sel_retrain":  sel_retrain,  "sel_ifru":  sel_ifru,
        "pfr_unlearn":   pfr_unlearn,  "pfr_retrain":  pfr_retrain,  "pfr_ifru":  pfr_ifru,
        "ov_unlearn_rt": ov_unlearn_rt, "ov_ifru_rt":  ov_ifru_rt,
    })

df_res = pd.DataFrame(results)

print(f"\n[汇总] 三方对比（n={len(df_res)} 用户）")
print("─" * 78)
for label, col in [
    ("Target Score Drop   (Unlearn)", "drop_unlearn"),
    ("Target Score Drop   (Retrain)", "drop_retrain"),
    ("Target Score Drop   (IFRU)   ", "drop_ifru"),
    ("Non-target Change   (Unlearn)", "nt_unlearn"),
    ("Non-target Change   (Retrain)", "nt_retrain"),
    ("Non-target Change   (IFRU)   ", "nt_ifru"),
    ("Selectivity Ratio   (Unlearn)", "sel_unlearn"),
    ("Selectivity Ratio   (Retrain)", "sel_retrain"),
    ("Selectivity Ratio   (IFRU)   ", "sel_ifru"),
    ("PFR_multi           (Unlearn)", "pfr_unlearn"),
    ("PFR_multi           (Retrain)", "pfr_retrain"),
    ("PFR_multi           (IFRU)   ", "pfr_ifru"),
    (f"Overlap@{TOP_K}  Unlearn vs Retrain", "ov_unlearn_rt"),
    (f"Overlap@{TOP_K}  IFRU    vs Retrain", "ov_ifru_rt"),
]:
    vals = df_res[col].dropna()
    print(f"  {label:<36}  mean={vals.mean():.4f}, std={vals.std():.4f}  (n={len(vals)})")


# ══════════════════════════════════════════════════════════════════════════════
# 10. 保存
# ══════════════════════════════════════════════════════════════════════════════
torch.save({
    "user_emb_ifru":  user_emb_ifru,
    "item_emb_ifru":  item_emb_ifru,
    "phi_user_idx":   phi_user_idx,
    "phi_item_idx":   phi_item_idx,
    "t_star":         t_star.numpy(),
    "I_solved":       I_solved.numpy(),
    "n_train":        n_train,
    "hyperparams":    {
        "PRUNE_A0": PRUNE_A0, "PRUNE_A1": PRUNE_A1,
        "LR_INFLUENCE": LR_INFLUENCE, "N_ITERS": N_ITERS,
        "N_HVP_BATCHES": N_HVP_BATCHES, "HVP_BATCH_SIZE": HVP_BATCH_SIZE,
    },
}, os.path.join(SAVE_DIR, "ifru_results.pt"))

df_res.to_csv(os.path.join(SAVE_DIR, "ifru_metrics.csv"), index=False)
print(f"\n[完成] 结果保存至 {SAVE_DIR}/")

[配置] 完成

[1] 加载数据...
  n_users=27794, n_items=144853, n_train=363218

[2] 构建 hat_A(D)...
  hat_A: nnz=722798

[3] 构建 D_remove...
  |D_remove|=3696, |V(D_r)|: 889 用户 / 2144 物品

[4] 构建 hat_A_{-r} 与 D_c...


/var/folders/rz/8jmyj8817yz96t1prby58jc40000gn/T/ipykernel_95891/1709469059.py:156: RuntimeWarning: divide by zero encountered in divide
  ds = np.where(dw > 0, 1.0 / np.sqrt(dw), 0.0)


  |D_retrain|=359497, |D_c|=78525

[5] 重要性剪枝（论文 Algorithm 1）...
  φ: 14383 用户 + 6251 物品 = 1320576 参数

[6] 计算 t* = -∇_φ L_d - ∇_φ L_s (Eq.13)...
  L_d=14.3898, L_s=-7.2253, ||t*||=4.4422

[7] HVP + Adam 求解影响函数 (Eq.15-16)...
  Adam lr=1e+00, iters=100, HVP batches=3×2048
  Iter   1/100  ||g_t||=4.4422  ||t||=1149.0470  [0.8s]
  Iter  10/100  ||g_t||=5.9072  ||t||=711.3299  [7.3s]
  Iter  20/100  ||g_t||=2.1403  ||t||=537.8644  [14.2s]
  Iter  30/100  ||g_t||=1.1404  ||t||=460.8235  [21.2s]
  Iter  40/100  ||g_t||=1.0006  ||t||=451.9417  [28.2s]
  Iter  50/100  ||g_t||=0.6672  ||t||=449.0256  [35.0s]
  Iter  60/100  ||g_t||=0.3943  ||t||=448.2182  [42.0s]
  Iter  70/100  ||g_t||=0.2121  ||t||=445.8007  [48.9s]
  Iter  80/100  ||g_t||=0.1072  ||t||=444.9534  [55.8s]
  Iter  90/100  ||g_t||=0.0719  ||t||=444.3329  [62.8s]
  Iter 100/100  ||g_t||=0.0517  ||t||=444.0913  [69.9s]

  求解完成：||I||=444.0913

[8] 参数更新 (Eq.14) 与 embedding 提取...
  user_emb_ifru: (27794, 64), item_emb_ifru: (144853, 64